# Brazilian E-Commerce Analysis — Schema Exploration

This notebook loads the Olist dataset, inspects all tables, documents table grain, and prepares the SQLite database for the SQL business analysis.

I'm not answering business questions yet here. First I need to understand the tables, keys, missing values, and row grain, because bad joins in a multi-table dataset can create wrong metrics.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DB_PATH = PROCESSED_DIR / "olist.db"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

In [3]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "geolocation": geolocation,
    "category_translation": category_translation
}

## 1. Table Overview

First I checked table sizes and full duplicate rows to get a quick sense of the dataset before looking at keys.

In [4]:
overview = []

for name, df in tables.items():
    overview.append({
        "table_name": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum()
    })

overview_df = pd.DataFrame(overview)
overview_df

,table_name,rows,columns,duplicate_rows
0,customers,99441,5,0
1,orders,99441,8,0
2,order_items,112650,7,0
3,products,32951,9,0
4,sellers,3095,4,0
5,payments,103886,5,0
6,reviews,99224,7,0
7,geolocation,1000163,5,261831
8,category_translation,71,2,0


### Table Overview Observations

- Most tables do not contain full duplicate rows.
- The geolocation table contains a large number of duplicate rows. This is expected because location records can repeat for the same zip-code prefix, city, or coordinate combination.
- The orders, customers, products, sellers, payments, reviews, and order_items tables appear structurally clean from a full-row duplicate perspective.
- Full-row duplicate checks are only a first pass. I still need key-level checks because a table can have no duplicate rows while still having repeated IDs.

## 2. Missing Values Overview

This section checks missing values by table and by column. I checked missing values separately because missingness depends on the column, not just the table.

In [5]:
missing_summary = []

for name, df in tables.items():
    missing_counts = df.isna().sum()
    missing_percentages = (missing_counts / len(df) * 100).round(2)

    for column in df.columns:
        if missing_counts[column] > 0:
            missing_summary.append({
                "table_name": name,
                "column_name": column,
                "missing_count": missing_counts[column],
                "missing_percentage": missing_percentages[column]
            })

missing_summary_df = pd.DataFrame(missing_summary)
missing_summary_df.sort_values(
    by=["table_name", "missing_count"],
    ascending=[True, False]
)

,table_name,column_name,missing_count,missing_percentage
2,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
0,orders,order_approved_at,160,0.16
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,products,product_weight_g,2,0.01
8,products,product_length_cm,2,0.01
9,products,product_height_cm,2,0.01


## 3. Key Uniqueness Checks

This section checks whether important identifier columns behave like unique keys.

In [6]:
key_uniqueness_checks = {
    "customers.customer_id": customers["customer_id"].is_unique,
    "customers.customer_unique_id": customers["customer_unique_id"].is_unique,
    "orders.order_id": orders["order_id"].is_unique,
    "products.product_id": products["product_id"].is_unique,
    "sellers.seller_id": sellers["seller_id"].is_unique,
    "reviews.review_id": reviews["review_id"].is_unique,
}

key_uniqueness_df = pd.DataFrame(
    key_uniqueness_checks.items(),
    columns=["key_column", "is_unique"]
)

key_uniqueness_df

,key_column,is_unique
0,customers.customer_id,True
1,customers.customer_unique_id,False
2,orders.order_id,True
3,products.product_id,True
4,sellers.seller_id,True
5,reviews.review_id,False


### Key Uniqueness Observation

- `customer_id` is unique, while `customer_unique_id` is not. This means that `customer_id` identifies a specific order/customer record, while `customer_unique_id` can appear across multiple purchases by the same real customer. These repeated values should not be dropped because they may be useful for repeat-customer analysis.
- `review_id` is not unique, so it should not be assumed to be a primary key by itself. I shouldn't treat `review_id` as a primary key by itself. The duplicated review IDs need to be checked before using reviews in joins.

## 4. Duplicate Key Counts

This section counts duplicated values in important key columns. This is more informative than only checking whether a column is unique.

In [7]:
key_checks = {
    "customers.customer_id": customers["customer_id"].duplicated().sum(),
    "customers.customer_unique_id": customers["customer_unique_id"].duplicated().sum(),
    "orders.order_id": orders["order_id"].duplicated().sum(),
    "products.product_id": products["product_id"].duplicated().sum(),
    "sellers.seller_id": sellers["seller_id"].duplicated().sum(),
    "reviews.review_id": reviews["review_id"].duplicated().sum(),
    "reviews.order_id": reviews["order_id"].duplicated().sum(),
}

key_checks_df = pd.DataFrame(
    key_checks.items(),
    columns=["key_column", "duplicated_values"]
)

key_checks_df

,key_column,duplicated_values
0,customers.customer_id,0
1,customers.customer_unique_id,3345
2,orders.order_id,0
3,products.product_id,0
4,sellers.seller_id,0
5,reviews.review_id,814
6,reviews.order_id,551


### Key Uniqueness Observations

* `customer_id`, `order_id`, `product_id`, and `seller_id` do not contain duplicated values in their respective tables, so they appear to behave like unique identifiers.
* `customer_unique_id` contains duplicated values. This is expected because it represents the same real customer across multiple purchases, while `customer_id` identifies a specific customer/order record.
* `review_id` and `reviews.order_id` both contain duplicated values, so the reviews table needs inspection before assuming a single-row-per-order or single-row-per-review structure.

## 5. Composite Key Checks

Some tables naturally need more than one column to identify a row, so I checked likely composite keys.

In [8]:
composite_key_checks = {
    "order_items.order_id + order_item_id": order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum(),

    "payments.order_id + payment_sequential": payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum(),

    "reviews.review_id + order_id": reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum(),
}

composite_key_checks_df = pd.DataFrame(
    composite_key_checks.items(),
    columns=["composite_key", "duplicated_rows"]
)

composite_key_checks_df

,composite_key,duplicated_rows
0,order_items.order_id + order_item_id,0
1,payments.order_id + payment_sequential,0
2,reviews.review_id + order_id,0


### Composite Key Observations

* The `order_items` table has a unique `order_id + order_item_id` combination, which suggests that each row represents one item within an order.
* The `payments` table has a unique `order_id + payment_sequential` combination, which means that each row represents one payment record or payment sequence for an order.
* The `reviews` table has a unique `review_id + order_id` combination, even though `review_id` and `order_id` are not unique individually. This means the reviews table should not be treated as one-row-per-review-id or one-row-per-order without further care.

## 6. Table Grain Summary

This was the most important part of the schema exploration for me: knowing what one row represents before joining tables.

- `customers`: one row represents one customer/order-level customer record. `customer_id` is unique, while `customer_unique_id` can repeat across multiple purchases by the same real customer.

- `orders`: one row represents one order. `order_id` is unique.

- `order_items`: one row represents one item inside an order. An order can have multiple item rows, so the natural row identifier is `order_id + order_item_id`.

- `products`: one row represents one product. `product_id` is unique.

- `sellers`: one row represents one seller. `seller_id` is unique.

- `payments`: one row represents one payment record or payment sequence for an order. An order can have multiple payment rows, so the natural row identifier is `order_id + payment_sequential`.

- `reviews`: one row represents a review record linked to an order. `review_id` and `order_id` are not unique individually, but `review_id + order_id` is unique.

- `geolocation`: one row represents a zip-code/location record. This table contains many duplicate rows and will require separate handling if used for location analysis.

- `category_translation`: one row represents a product category name and its English translation.

## 7. Data Type and Date Column Inspection

I inspected the order date columns because they will be used later for trends, delivery time, and late-delivery analysis.

In [9]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [10]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [11]:
for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

### Date Column Observations

- All main order timeline columns were successfully converted to datetime format.
- `order_purchase_timestamp` and `order_estimated_delivery_date` have no missing values.
- `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` contain missing values.
- Missing approval or delivery timestamps likely correspond to orders that were canceled, unavailable, or never completed.
- These columns will be useful later for delivery-time, delay, and order trend analysis.

## 8. Delivery Date Missingness by Order Status

Before calculating delivery time, missing delivery dates need to be inspected by order status.

In [12]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
orders.loc[
    orders["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

### Delivery Date Missingness Observation

Most missing customer delivery dates belong to orders that were not completed, such as shipped, canceled, unavailable, invoiced, or processing orders. This suggests that missing delivery dates are mostly meaningful and related to order status rather than random missing data.

However, a small number of orders are marked as delivered while still missing `order_delivered_customer_date`. These rows may require handling later when calculating delivery-time metrics.

## 9. SQLite Database Setup

The cleaned table objects are written into a local SQLite database. This database is used in the SQL business analysis notebook.

In [14]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

In [15]:
for table_name, df in tables.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)

In [16]:
pd.read_sql_query("""
SELECT COUNT(*) AS total_orders
FROM orders;
""", conn)

,total_orders
0,99441


In [17]:
pd.read_sql_query("""
SELECT 
    order_status,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
""", conn)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


### SQL Setup Observation

The SQLite setup worked. I can query the `orders` table from the notebook, so the database is ready for the SQL analysis.

## Schema Exploration Summary

This notebook confirmed the main structure of the Olist dataset and prepared the SQLite database for analysis.

The most important takeaways are:

- `orders`, `customers`, `products`, and `sellers` have clear unique identifiers.
- `order_items`, `payments`, and `reviews` require composite-key thinking.
- `customer_unique_id` can repeat because it represents the same real customer across multiple purchases.
- The geolocation table contains many duplicate rows and should be handled carefully if used later.
- Missing delivery timestamps are mostly connected to order status rather than random missingness.
- The SQLite database is ready for the main SQL business analysis.